<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/The_Trade_Off_Between_Cost%2C_Lead_Time_and_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# GLOBAL SOURCING TRADE-OFF ANALYSIS
# ================================================================
#
# OBJETIVO DEL EJERCICIO
#
# Comparar tres estrategias de abastecimiento:
#
# 1. GLOBAL SUPPLIER
# 2. REGIONAL SUPPLIER
# 3. DUAL SOURCING
#
# para observar el trade-off entre:
#
# - Cost
# - Lead Time
# - Inventory
# - Service Level
# - Risk
# - Total Cost of Ownership
#
# La idea central es:
#
# "The cheapest supplier is not always the best sourcing strategy."
#
# Este ejercicio NO busca que los estudiantes programen desde cero.
# El objetivo es que:
#
# 1. Ejecuten el modelo.
# 2. Analicen las gráficas.
# 3. Comparen escenarios.
# 4. Expliquen cuál estrategia elegirían.
#
# ================================================================


# ================================================================
# 1. IMPORTAR LIBRERÍAS
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ================================================================
# 2. DEFINIR DATOS GENERALES DEL NEGOCIO
# ================================================================
#
# Vamos a asumir que la empresa necesita satisfacer
# una demanda anual relativamente estable.
#
# ================================================================


ANNUAL_DEMAND = 120000

# Demanda anual del producto.
# Ejemplo:
# la empresa vende o consume 120.000 unidades al año.


DAILY_DEMAND = ANNUAL_DEMAND / 365

# Convertimos la demanda anual en demanda diaria.
#
# Esto será útil para calcular:
#
# - inventario durante el lead time
# - safety stock
# - cobertura
# - riesgo de faltantes


HOLDING_RATE = 0.22

# Tasa anual de mantenimiento del inventario.
#
# 22% significa que mantener inventario cuesta aproximadamente
# 22% de su valor por año.
#
# Aquí se incluyen costos como:
#
# - almacenamiento
# - capital inmovilizado
# - seguros
# - deterioro
# - obsolescencia
# - manipulación


STOCKOUT_COST_PER_UNIT = 18

# Costo estimado asociado a una unidad que no puede atenderse.
#
# Puede representar:
#
# - pérdida de margen
# - penalización
# - venta perdida
# - compra urgente
# - impacto en servicio


# ================================================================
# 3. DEFINIR LOS TRES ESCENARIOS
# ================================================================
#
# Los datos son ficticios y tienen fines pedagógicos.
#
# ================================================================


data = {

    "Strategy": [
        "Global Supplier",
        "Regional Supplier",
        "Dual Sourcing"
    ],

    # ------------------------------------------------------------
    # PURCHASE PRICE
    #
    # Precio unitario de compra.
    #
    # El proveedor global parece inicialmente más atractivo.
    # ------------------------------------------------------------

    "Unit_Price": [
        8.50,
        10.20,
        9.10
    ],

    # ------------------------------------------------------------
    # LEAD TIME
    #
    # Días promedio desde la orden hasta la recepción.
    #
    # ------------------------------------------------------------

    "Lead_Time_Days": [
        55,
        12,
        30
    ],

    # ------------------------------------------------------------
    # LEAD TIME VARIABILITY
    #
    # Variabilidad alrededor del tiempo esperado.
    #
    # Mayor valor = más incertidumbre.
    #
    # ------------------------------------------------------------

    "Lead_Time_Variability": [
        14,
        4,
        8
    ],

    # ------------------------------------------------------------
    # ON-TIME DELIVERY
    #
    # Porcentaje de entregas realizadas a tiempo.
    #
    # ------------------------------------------------------------

    "On_Time_Delivery": [
        87,
        97,
        94
    ],

    # ------------------------------------------------------------
    # TRANSPORT COST
    #
    # Costo logístico unitario.
    #
    # El proveedor global puede requerir:
    #
    # - marítimo
    # - puertos
    # - seguros
    # - manipulación
    #
    # ------------------------------------------------------------

    "Transport_Cost_Per_Unit": [
        1.20,
        0.55,
        0.90
    ],

    # ------------------------------------------------------------
    # DUTIES / TARIFFS
    #
    # Aranceles u otros costos unitarios.
    #
    # ------------------------------------------------------------

    "Duties_Per_Unit": [
        0.65,
        0.10,
        0.40
    ],

    # ------------------------------------------------------------
    # DISRUPTION PROBABILITY
    #
    # Probabilidad anual estimada de una disrupción importante.
    #
    # Ejemplo:
    # 0.18 = 18%
    #
    # ------------------------------------------------------------

    "Disruption_Probability": [
        0.18,
        0.06,
        0.09
    ],

    # ------------------------------------------------------------
    # DISRUPTION IMPACT
    #
    # Costo estimado si la disrupción ocurre.
    #
    # ------------------------------------------------------------

    "Disruption_Impact": [
        240000,
        90000,
        130000
    ],

    # ------------------------------------------------------------
    # SERVICE LEVEL
    #
    # Nivel de servicio esperado.
    #
    # ------------------------------------------------------------

    "Service_Level": [
        91,
        98,
        96
    ],

    # ------------------------------------------------------------
    # STOCKOUT RATE
    #
    # Porcentaje de demanda anual potencialmente
    # afectada por faltantes.
    #
    # ------------------------------------------------------------

    "Stockout_Rate": [
        0.035,
        0.010,
        0.018
    ],

    # ------------------------------------------------------------
    # RISK SCORE
    #
    # Indicador sintético 0-100.
    #
    # Mayor valor = mayor riesgo.
    #
    # Considera de forma simplificada:
    #
    # - distancia
    # - transporte
    # - dependencia
    # - vulnerabilidad geográfica
    # - incertidumbre
    #
    # ------------------------------------------------------------

    "Risk_Score": [
        78,
        28,
        46
    ]
}


df = pd.DataFrame(data)


# ================================================================
# 4. CALCULAR INVENTARIO DURANTE EL LEAD TIME
# ================================================================
#
# Si consumimos aproximadamente DAILY_DEMAND unidades por día,
# durante el lead time necesitamos cubrir:
#
# Daily Demand x Lead Time
#
# ================================================================


df["Lead_Time_Inventory"] = (

    DAILY_DEMAND

    *

    df["Lead_Time_Days"]

)


# ================================================================
# 5. CALCULAR SAFETY STOCK
# ================================================================
#
# Para mantener el ejercicio comprensible,
# utilizaremos una aproximación simple:
#
# Safety Stock =
#
# Daily Demand
#
# x
#
# Lead Time Variability
#
# x
#
# Risk Adjustment
#
#
# Estrategias más riesgosas tendrán un mayor factor.
#
# ================================================================


risk_adjustment = {

    "Global Supplier": 1.40,

    "Regional Supplier": 0.80,

    "Dual Sourcing": 1.00

}


df["Risk_Adjustment"] = (

    df["Strategy"]
    .map(risk_adjustment)

)


df["Safety_Stock"] = (

    DAILY_DEMAND

    *

    df["Lead_Time_Variability"]

    *

    df["Risk_Adjustment"]

)


# ================================================================
# 6. CALCULAR INVENTARIO PROMEDIO
# ================================================================
#
# Para simplificar:
#
# Average Inventory =
#
# 50% del inventario durante lead time
#
# +
#
# Safety Stock
#
#
# No representa una fórmula universal,
# sino una aproximación pedagógica.
#
# ================================================================


df["Average_Inventory"] = (

    df["Lead_Time_Inventory"] * 0.50

    +

    df["Safety_Stock"]

)


# ================================================================
# 7. CALCULAR COSTO DE COMPRA ANUAL
# ================================================================
#
# Purchase Cost =
#
# Annual Demand x Unit Price
#
# ================================================================


df["Annual_Purchase_Cost"] = (

    ANNUAL_DEMAND

    *

    df["Unit_Price"]

)


# ================================================================
# 8. CALCULAR COSTO DE TRANSPORTE
# ================================================================


df["Annual_Transport_Cost"] = (

    ANNUAL_DEMAND

    *

    df["Transport_Cost_Per_Unit"]

)


# ================================================================
# 9. CALCULAR ARANCELES
# ================================================================


df["Annual_Duties"] = (

    ANNUAL_DEMAND

    *

    df["Duties_Per_Unit"]

)


# ================================================================
# 10. CALCULAR VALOR DEL INVENTARIO
# ================================================================
#
# Inventory Value =
#
# Average Inventory x Unit Price
#
# ================================================================


df["Inventory_Value"] = (

    df["Average_Inventory"]

    *

    df["Unit_Price"]

)


# ================================================================
# 11. CALCULAR CARRYING COST
# ================================================================
#
# Carrying Cost =
#
# Inventory Value x Holding Rate
#
# ================================================================


df["Inventory_Carrying_Cost"] = (

    df["Inventory_Value"]

    *

    HOLDING_RATE

)


# ================================================================
# 12. CALCULAR COSTO DE STOCKOUTS
# ================================================================
#
# Primero estimamos cuántas unidades pueden quedar
# afectadas por faltantes.
#
# ================================================================


df["Expected_Stockout_Units"] = (

    ANNUAL_DEMAND

    *

    df["Stockout_Rate"]

)


# Luego calculamos su costo.

df["Expected_Stockout_Cost"] = (

    df["Expected_Stockout_Units"]

    *

    STOCKOUT_COST_PER_UNIT

)


# ================================================================
# 13. CALCULAR COSTO ESPERADO DE DISRUPCIÓN
# ================================================================
#
# Expected Disruption Cost =
#
# Probability x Impact
#
#
# Ejemplo:
#
# 18% x $240,000
#
# No significa que necesariamente vamos a gastar ese valor.
# Es una forma de incorporar riesgo esperado al análisis.
#
# ================================================================


df["Expected_Disruption_Cost"] = (

    df["Disruption_Probability"]

    *

    df["Disruption_Impact"]

)


# ================================================================
# 14. CALCULAR TOTAL COST OF OWNERSHIP
# ================================================================
#
# TCO incluye:
#
# Purchase Cost
# Transport
# Duties
# Inventory Carrying Cost
# Stockout Cost
# Expected Disruption Cost
#
# ================================================================


df["TCO"] = (

    df["Annual_Purchase_Cost"]

    +

    df["Annual_Transport_Cost"]

    +

    df["Annual_Duties"]

    +

    df["Inventory_Carrying_Cost"]

    +

    df["Expected_Stockout_Cost"]

    +

    df["Expected_Disruption_Cost"]

)


# ================================================================
# 15. CALCULAR TCO POR UNIDAD
# ================================================================


df["TCO_Per_Unit"] = (

    df["TCO"]

    /

    ANNUAL_DEMAND

)


# ================================================================
# 16. MOSTRAR TABLA DE RESULTADOS
# ================================================================


summary = df[

    [
        "Strategy",
        "Unit_Price",
        "Lead_Time_Days",
        "Safety_Stock",
        "Average_Inventory",
        "Service_Level",
        "Risk_Score",
        "TCO",
        "TCO_Per_Unit"
    ]

].copy()


print("\n" + "="*100)

print("GLOBAL SOURCING STRATEGY RESULTS")

print("="*100)


print(

    summary
    .round(2)
    .to_string(index=False)

)


# ================================================================
# 17. GRÁFICA 1
#
# PURCHASE PRICE VS TCO PER UNIT
# ================================================================
#
# Esta gráfica permite visualizar una idea clave:
#
# el proveedor con menor precio de compra
# puede no tener el menor costo total.
#
# ================================================================


x = np.arange(
    len(df)
)


width = 0.35


plt.figure(
    figsize=(11,7)
)


plt.bar(

    x - width/2,

    df["Unit_Price"],

    width,

    label="Purchase Price",

    color="#3498DB"

)


plt.bar(

    x + width/2,

    df["TCO_Per_Unit"],

    width,

    label="TCO per Unit",

    color="#E67E22"

)


plt.xticks(

    x,

    df["Strategy"]

)


plt.ylabel(
    "Cost per Unit"
)


plt.title(

    "Purchase Price vs Total Cost of Ownership",

    fontsize=18,

    fontweight="bold"

)


plt.legend()


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# INTERPRETACIÓN GRÁFICA 1
# ================================================================
#
# Observe:
#
# Global Supplier puede tener el precio inicial más bajo.
#
# Sin embargo, cuando agregamos:
#
# transporte
# aranceles
# inventario
# faltantes
# disrupciones
#
# la diferencia real puede reducirse significativamente.
#
# Esta es precisamente la lógica de TCO.
#
# ================================================================


# ================================================================
# 18. GRÁFICA 2
#
# LEAD TIME VS AVERAGE INVENTORY
# ================================================================
#
# Esta gráfica permite observar:
#
# mayor lead time
#
#        ↓
#
# mayor necesidad de inventario
#
# ================================================================


plt.figure(
    figsize=(11,7)
)


colors = [
    "#E74C3C",
    "#2ECC71",
    "#F39C12"
]


plt.scatter(

    df["Lead_Time_Days"],

    df["Average_Inventory"],

    s=500,

    color=colors,

    edgecolor="black",

    alpha=0.80

)


for i, row in df.iterrows():

    plt.text(

        row["Lead_Time_Days"] + 1,

        row["Average_Inventory"] + 100,

        row["Strategy"],

        fontsize=10

    )


plt.xlabel(
    "Lead Time (Days)"
)


plt.ylabel(
    "Average Inventory (Units)"
)


plt.title(

    "Lead Time vs Inventory Requirement",

    fontsize=18,

    fontweight="bold"

)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# INTERPRETACIÓN GRÁFICA 2
# ================================================================
#
# El proveedor global requiere mayor inventario
# porque tarda más tiempo en abastecer.
#
# Esto significa:
#
# más capital de trabajo
# más espacio
# mayor exposición a obsolescencia
# mayor carrying cost
#
# ================================================================


# ================================================================
# 19. GRÁFICA 3
#
# SERVICE LEVEL VS RISK
# ================================================================
#
# Una estrategia ideal tendría:
#
# HIGH SERVICE
#
# y
#
# LOW RISK
#
# ================================================================


plt.figure(
    figsize=(11,7)
)


plt.scatter(

    df["Risk_Score"],

    df["Service_Level"],

    s=df["TCO_Per_Unit"] * 70,

    color=colors,

    edgecolor="black",

    alpha=0.75

)


for i, row in df.iterrows():

    plt.text(

        row["Risk_Score"] + 1,

        row["Service_Level"] + 0.2,

        row["Strategy"],

        fontsize=10

    )


plt.xlabel(
    "Supply Risk Score"
)


plt.ylabel(
    "Service Level (%)"
)


plt.title(

    "Risk vs Service Level",

    fontsize=18,

    fontweight="bold"

)


plt.axvline(

    50,

    linestyle="--",

    color="#E74C3C",

    alpha=0.60

)


plt.axhline(

    95,

    linestyle="--",

    color="#2ECC71",

    alpha=0.60

)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# INTERPRETACIÓN GRÁFICA 3
# ================================================================
#
# Una posición deseable sería:
#
# parte superior izquierda
#
# porque representa:
#
# menor riesgo
# mayor nivel de servicio
#
# El proveedor global puede ser atractivo en precio,
# pero aparece más expuesto al riesgo.
#
# ================================================================


# ================================================================
# 20. GRÁFICA 4
#
# COMPOSICIÓN DEL TOTAL COST OF OWNERSHIP
# ================================================================
#
# Esta gráfica permite ver qué está causando realmente
# el costo total de cada estrategia.
#
# ================================================================


cost_components = [

    "Annual_Purchase_Cost",

    "Annual_Transport_Cost",

    "Annual_Duties",

    "Inventory_Carrying_Cost",

    "Expected_Stockout_Cost",

    "Expected_Disruption_Cost"

]


labels = [

    "Purchase",

    "Transport",

    "Duties",

    "Inventory",

    "Stockouts",

    "Disruption"

]


component_colors = [

    "#3498DB",

    "#9B59B6",

    "#F1C40F",

    "#1ABC9C",

    "#E67E22",

    "#E74C3C"

]


bottom = np.zeros(
    len(df)
)


plt.figure(
    figsize=(12,8)
)


for component, label, color in zip(

    cost_components,

    labels,

    component_colors

):

    plt.bar(

        df["Strategy"],

        df[component],

        bottom=bottom,

        label=label,

        color=color

    )


    bottom += df[component].values


plt.title(

    "Total Cost of Ownership Breakdown",

    fontsize=18,

    fontweight="bold"

)


plt.ylabel(
    "Annual Cost"
)


plt.legend(
    title="Cost Component"
)


plt.grid(
    axis="y",
    alpha=0.15
)


plt.tight_layout()

plt.show()


# ================================================================
# INTERPRETACIÓN GRÁFICA 4
# ================================================================
#
# Esta gráfica es muy importante.
#
# Permite que los estudiantes vean que:
#
# Purchase Price
#
# es solamente UNA PARTE del costo total.
#
#
# Dos proveedores pueden parecer muy diferentes
# cuando observamos únicamente el precio,
#
# pero mucho más similares cuando analizamos TCO.
#
# ================================================================


# ================================================================
# 21. GRÁFICA 5
#
# COST - SERVICE - RISK TRADE-OFF
# ================================================================
#
# Vamos a construir una gráfica con:
#
# X = TCO por unidad
#
# Y = Service Level
#
# Tamaño burbuja = Risk Score
#
#
# La mejor estrategia idealmente tendría:
#
# bajo costo
# alto servicio
# bajo riesgo
#
# ================================================================


plt.figure(
    figsize=(12,8)
)


plt.scatter(

    df["TCO_Per_Unit"],

    df["Service_Level"],

    s=df["Risk_Score"] * 12,

    color=colors,

    alpha=0.75,

    edgecolor="black"

)


for i, row in df.iterrows():

    plt.text(

        row["TCO_Per_Unit"] + 0.03,

        row["Service_Level"] + 0.15,

        row["Strategy"],

        fontsize=10

    )


plt.xlabel(
    "Total Cost of Ownership per Unit"
)


plt.ylabel(
    "Service Level (%)"
)


plt.title(

    "Cost-Service-Risk Trade-Off",

    fontsize=19,

    fontweight="bold"

)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# INTERPRETACIÓN GRÁFICA 5
# ================================================================
#
# Esta es la gráfica gerencial más importante.
#
# Permite responder:
#
# ¿Cuál estrategia ofrece el mejor equilibrio?
#
#
# Una estrategia con costo muy bajo,
# pero servicio bajo y riesgo alto,
# puede no ser conveniente.
#
#
# Una estrategia muy segura,
# pero extremadamente costosa,
# tampoco necesariamente es óptima.
#
#
# Por eso buscamos BALANCE.
#
# ================================================================


# ================================================================
# 22. CREAR SCORE GERENCIAL DE BALANCE
# ================================================================
#
# Solo con fines pedagógicos,
# podemos crear un índice que combine:
#
# Cost
# Service
# Risk
#
# ================================================================


# Primero normalizamos TCO.
#
# Un TCO menor debe producir mayor puntuación.

df["Cost_Score"] = (

    100

    -

    (
        (df["TCO_Per_Unit"] - df["TCO_Per_Unit"].min())

        /

        (df["TCO_Per_Unit"].max() - df["TCO_Per_Unit"].min())

        * 100
    )

)


# Service Level ya está aproximadamente en escala 0-100.

df["Service_Score"] = df["Service_Level"]


# Menor riesgo = mayor puntuación.

df["Resilience_Score"] = (

    100 - df["Risk_Score"]

)


# ================================================================
# PESOS DEL SCORE
# ================================================================
#
# Costo       = 40%
# Servicio    = 35%
# Resiliencia = 25%
#
# Estos pesos pueden modificarse.
#
# ================================================================


df["Balance_Score"] = (

    df["Cost_Score"] * 0.40

    +

    df["Service_Score"] * 0.35

    +

    df["Resilience_Score"] * 0.25

)


# ================================================================
# 23. RANKING FINAL
# ================================================================


ranking = (

    df[
        [
            "Strategy",
            "Cost_Score",
            "Service_Score",
            "Resilience_Score",
            "Balance_Score"
        ]
    ]

    .sort_values(
        "Balance_Score",
        ascending=False
    )

)


print("\n" + "="*100)

print("FINAL SOURCING STRATEGY RANKING")

print("="*100)


print(

    ranking
    .round(2)
    .to_string(index=False)

)


# ================================================================
# 24. GRÁFICA FINAL DE RANKING
# ================================================================


ranking_plot = ranking.sort_values(
    "Balance_Score"
)


final_colors = [

    "#E74C3C",
    "#F39C12",
    "#2ECC71"
]


plt.figure(
    figsize=(10,6)
)


bars = plt.barh(

    ranking_plot["Strategy"],

    ranking_plot["Balance_Score"],

    color=final_colors

)


plt.xlabel(
    "Strategic Balance Score"
)


plt.title(

    "Final Global Sourcing Strategy Ranking",

    fontsize=18,

    fontweight="bold"

)


for bar in bars:

    width = bar.get_width()

    plt.text(

        width + 1,

        bar.get_y()
        + bar.get_height()/2,

        f"{width:.1f}",

        va="center",

        fontweight="bold"

    )


plt.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# 25. INTERPRETACIÓN AUTOMÁTICA
# ================================================================


best_strategy = ranking.iloc[0]["Strategy"]

best_score = ranking.iloc[0]["Balance_Score"]


print("\n" + "="*100)

print("MANAGEMENT INTERPRETATION")

print("="*100)


print(

    f"""
Based on the assumptions used in this model,
the strategy with the strongest balance between:

- Cost
- Service
- Risk

is:

{best_strategy}

Strategic Balance Score:

{best_score:.2f}

IMPORTANT:

This does NOT mean that the strategy will always be the best.

The result depends on:

- demand
- costs
- risk probabilities
- service requirements
- lead times
- managerial priorities

Changing these assumptions may change the final decision.
"""

)


# ================================================================
# 26. PREGUNTAS PARA LOS ESTUDIANTES
# ================================================================


print("\n" + "="*100)

print("MANAGEMENT DISCUSSION QUESTIONS")

print("="*100)


print("""

1. Which strategy has the lowest purchase price?

2. Is the strategy with the lowest purchase price
   also the strategy with the lowest TCO?

3. Which strategy requires the highest inventory?

4. How does lead time influence working capital?

5. Which strategy offers the highest service level?

6. Which strategy presents the greatest disruption risk?

7. What hidden costs make the global supplier less attractive?

8. Is the regional supplier worth paying more for?

9. Why might dual sourcing provide a better balance?

10. If the company prioritized cost above everything else,
    which strategy would you select?

11. If the company prioritized resilience,
    which strategy would you select?

12. As Supply Chain Manager,
    which strategy would you recommend and why?

""")